# Explore a pretrained GPT-2 model

This optional lesson uses the smallest GPT-2 checkpoint through jadeGPT's nanoGPT-style implementation. No local training is required. The loader maps pretrained weights into the architecture derived from Andrej Karpathy's [nanoGPT](https://github.com/karpathy/nanoGPT); the original GPT-2 model is from [OpenAI](https://github.com/openai/gpt-2).

**You will:** explicitly enable a model download, inspect byte-pair tokenization, and vary a continuation prompt. GPT-2 is a base completion model, not a chat assistant. It continues text and can invent facts or repeat undesirable material from its training distribution.


## Environment and reproducibility

Use the environment described in the [README](README.md); Python 3.13 is the recommended starting point. From the repository root, run:

```sh
python -m pip install -e ".[notebooks]"
python -m jupyter lab
```

Choose the Python kernel from that environment and run cells **in order**. Installation is a one-time terminal step, not part of the lesson. Paths below work on Windows, macOS, and Linux. `auto` selects an available accelerator or CPU, then a supported precision. A seed makes comparisons more useful, but exact results can differ across hardware and library versions.


In [ ]:
from pathlib import Path
import json
import os
import sys

import torch
import jadegpt

ROOT = Path.cwd().resolve()
if not (ROOT / "jadegpt.py").is_file():
    raise RuntimeError("Start JupyterLab from the jadeGPT repository root.")

SEED = 1337
DEVICE = jadegpt.resolve_device("auto")
DTYPE = jadegpt.resolve_dtype("auto", DEVICE)
# A few CPU threads often suit a tiny model better than every available core.
if DEVICE == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(SEED)
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | {DEVICE} | {DTYPE}")


## 1. Opt into the download

The notebook runs without network activity by default. Set `ENABLE_PRETRAINED=True` to run the model cells. The first run downloads weights from Hugging Face and tokenizer data; later runs can reuse the local caches. The default `gpt2` has about 124 million parameters and needs roughly 500 MB for float32 weights alone. Loading can temporarily require more memory, and several GB of available RAM is a practical starting point.

CPU generation works but is slower than a GPU. Begin with one 80-token sample. `gpt2-medium`, `gpt2-large`, and `gpt2-xl` require substantially more resources; a larger checkpoint does not change this lesson's mechanism.


In [ ]:
ENABLE_PRETRAINED = False  # Change to True to permit first-use downloads and run inference.
PRETRAINED_MODEL = "gpt2"
model = None
if ENABLE_PRETRAINED:
    model = jadegpt.init_gpt2(gpt2_model=PRETRAINED_MODEL, random_seed=SEED)
    print(f"Loaded {PRETRAINED_MODEL}: context window {model.config.block_size} tokens")
else:
    print("Pretrained lesson skipped. Set ENABLE_PRETRAINED=True and re-run from this cell to begin.")


## 2. Inspect tokens before predicting them

GPT-2 uses byte-pair encoding rather than the scratch lesson's one-character vocabulary. Word fragments and spaces affect token boundaries; equal character counts need not produce equal token counts. The fixed GPT-2 tokenizer must accompany GPT-2 weights.

Tokenizer initialization may also need network access on first use, so it is covered by the same opt-in switch.


In [ ]:
PROMPT = "A good way to learn how a language model works is"
if ENABLE_PRETRAINED:
    import tiktoken

    tokenizer = tiktoken.get_encoding("gpt2")
    token_ids = tokenizer.encode(PROMPT)
    print(f"{len(PROMPT)} characters → {len(token_ids)} tokens")
    print([(token_id, tokenizer.decode([token_id])) for token_id in token_ids])
    assert tokenizer.decode(token_ids) == PROMPT


## 3. Generate a continuation

Sampling repeatedly appends one token to the prompt. The context window limits how much previous text the model can use. A lower temperature often sounds more predictable; a higher temperature increases variation. `top_k` filters unlikely tokens before the draw.

jadeGPT masks padded vocabulary entries when relevant so tokenizer decoding receives valid token IDs. The pretrained GPT-2 tokenizer is selected explicitly below; custom character metadata is not used for this model.


In [ ]:
if ENABLE_PRETRAINED:
    torch.manual_seed(SEED)
    output = jadegpt.generate_text(
        model=model, start=PROMPT, use_gpt2_encoding=True,
        meta_dir="", meta_file_name="",
        num_samples=1, max_new_tokens=80, temperature=0.8, top_k=40,
        device=DEVICE, dtype=DTYPE,
    )
    print(output)


## Experiments and next steps

- Keep the seed fixed and compare an unfinished sentence with a question. Does the model answer, continue, or imitate a different format?
- Compare temperatures of 0.5 and 1.0 while holding everything else fixed. Separate readability from factual reliability.
- Inspect tokenization for a leading space, punctuation, and non-English text. A token is not a universal unit of meaning.
- Open [finetune-gpt.ipynb](finetune-gpt.ipynb) and explicitly choose `SOURCE="gpt2"` to adapt pretrained weights to your own corpus.

Successful inference means the model loads and produces a continuation. It does not establish that the continuation is correct. Record the checkpoint name, prompt, seed, temperature, and `top_k` when sharing a reproducible comparison.
